# Student Health Risk: Shared OOF/CV Baseline

This notebook creates the validation foundation before model tuning.

- One shared stratified fold assignment
- Balanced-accuracy early stopping
- Matching OOF and test probabilities
- CatBoost, LightGBM, and XGBoost
- Fold scores, class recall, confusion matrices, and training curves
- EDA-derived ordinal, zero-exercise, low-calorie, and interaction features
- OOF subset diagnostics for stress, sleep, activity, missingness, and class boundaries
- Equal-probability ensemble as a diagnostic baseline

Start with smoke mode. Change only `RUN_MODE` to `full` after the smoke artifacts have
been checked.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "artifacts" / ".mplconfig"))

from src.oof_baseline import CVConfig, run_baseline

RUN_MODE = "smoke"  # Change to "full" only after smoke verification.
MODELS = ("catboost", "lightgbm", "xgboost")

config = CVConfig.for_mode(RUN_MODE)
config.models = MODELS
config

CVConfig(run_mode='smoke', models=('catboost', 'lightgbm', 'xgboost'), seed=20260704, n_splits=2, smoke_rows=60000, smoke_splits=2, class_weight_mode='balanced', output_dir='artifacts/oof_cv_baseline_smoke', log_period=50, early_stopping_rounds=80, catboost_iterations=500, lightgbm_iterations=700, xgboost_iterations=700)

## Run

The notebook prints every fold's row counts, validation balanced accuracy, and selected
iteration. All models use the same fold assignment and class order:

`fit`, `at-risk`, `unhealthy`.

In smoke mode, outputs are written to `artifacts/oof_cv_baseline_smoke/`. Full mode uses
`artifacts/oof_cv_baseline/`.

In [2]:
summary = run_baseline(ROOT, config)
summary

[15:16:19] Loading competition data
[15:16:19] train=(690088, 15), test=(295753, 14), target_share={'fit': 0.057678151192311705, 'at-risk': 0.8586745458550213, 'unhealthy': 0.08364730295266691}
[15:16:19] Smoke mode selected 60,000 stratified train rows
[15:16:19] Building shared feature views
[15:16:20] catboost fold 1/2: train=30,000, valid=30,000


/Users/parkyeonggon/Projects/kaggle/Kaggle_Predicting_Student_Health_Risk/.venv/lib/python3.14/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


0:	learn: 0.9208210	test: 0.9179798	best: 0.9179798 (0)	total: 78.5ms	remaining: 39.2s
50:	learn: 0.9406909	test: 0.9392514	best: 0.9392773 (48)	total: 902ms	remaining: 7.94s
100:	learn: 0.9433195	test: 0.9405466	best: 0.9405466 (100)	total: 1.74s	remaining: 6.87s
150:	learn: 0.9465859	test: 0.9420406	best: 0.9421814 (147)	total: 2.6s	remaining: 6s
200:	learn: 0.9482236	test: 0.9420363	best: 0.9425826 (172)	total: 3.47s	remaining: 5.16s
250:	learn: 0.9499186	test: 0.9419060	best: 0.9425826 (172)	total: 4.34s	remaining: 4.31s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.9425825753
bestIteration = 172

Shrink model to first 173 iterations.
[15:16:26] catboost fold 1: BAC=0.942583, best_iteration=173
[15:16:26] catboost fold 2/2: train=30,000, valid=30,000
0:	learn: 0.9223094	test: 0.9221123	best: 0.9221123 (0)	total: 19ms	remaining: 9.51s


/Users/parkyeonggon/Projects/kaggle/Kaggle_Predicting_Student_Health_Risk/.venv/lib/python3.14/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


50:	learn: 0.9404889	test: 0.9398166	best: 0.9398166 (50)	total: 850ms	remaining: 7.49s
100:	learn: 0.9442167	test: 0.9414019	best: 0.9414959 (99)	total: 1.68s	remaining: 6.65s
150:	learn: 0.9475589	test: 0.9420796	best: 0.9420926 (146)	total: 2.55s	remaining: 5.89s
200:	learn: 0.9507920	test: 0.9426729	best: 0.9428335 (184)	total: 3.41s	remaining: 5.07s
250:	learn: 0.9523571	test: 0.9423903	best: 0.9428335 (184)	total: 4.3s	remaining: 4.27s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.9428335387
bestIteration = 184

Shrink model to first 185 iterations.
[15:16:30] catboost fold 2: BAC=0.942834, best_iteration=185


...

/Users/parkyeonggon/Projects/kaggle/Kaggle_Predicting_Student_Health_Risk/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/parkyeonggon/Projects/kaggle/Kaggle_Predicting_Student_Health_Risk/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[15:17:17] xgboost full OOF balanced_accuracy=0.944928
[15:17:17] equal ensemble OOF balanced_accuracy=0.944460


/Users/parkyeonggon/Projects/kaggle/Kaggle_Predicting_Student_Health_Risk/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/parkyeonggon/Projects/kaggle/Kaggle_Predicting_Student_Health_Risk/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[15:17:17] Completed. Artifacts: /Users/parkyeonggon/Projects/kaggle/Kaggle_Predicting_Student_Health_Risk/artifacts/oof_cv_baseline_smoke
         model  oof_balanced_accuracy  fold_mean  fold_std  recall_fit  recall_at-risk  recall_unhealthy
       xgboost               0.944928   0.944928  0.000423    0.938746        0.937481          0.958557
equal_ensemble               0.944460        NaN       NaN    0.938168        0.937054          0.958159
      lightgbm               0.943124   0.943124  0.000729    0.934990        0.937616          0.956764
      catboost               0.942708   0.942708  0.000125    0.936146        0.931231          0.960749


,model,oof_balanced_accuracy,fold_mean,fold_std,recall_fit,recall_at-risk,recall_unhealthy
2,xgboost,0.944928,0.944928,0.000423,0.938746,0.937481,0.958557
3,equal_ensemble,0.944460,NaN,NaN,0.938168,0.937054,0.958159
1,lightgbm,0.943124,0.943124,0.000729,0.934990,0.937616,0.956764
0,catboost,0.942708,0.942708,0.000125,0.936146,0.931231,0.960749


## Required checks before full mode

1. Every model completed all smoke folds.
2. OOF and test probability rows sum to one.
3. `fold_scores.csv` reports all three class recalls.
4. The selected iteration follows validation balanced accuracy.
5. The submission IDs match `sample_submission.csv`.
6. Training curves show both balanced accuracy and logloss.
7. `subset_metrics.csv` identifies weak stress, sleep, activity, and boundary groups.

Do not tune stacking weights from public leaderboard feedback. The next stage will use
these OOF sources for repeated-CV model comparison, feature ablation, source diversity,
and class-wise stacking.